# ⚙️ Minimal OPRO: Optimization by PROmpting

**Exercise duration: ~6 minutes** | Run every cell top-to-bottom

---

## What is OPRO?

Instead of *you* crafting the perfect instruction, you let the **model propose its own instructions** and measure which works best.

**The loop:**
1. Start with seed instructions.
2. Score each on a small dataset (accuracy).
3. Build a **meta-prompt** showing past instructions + their scores.
4. Ask the model: *"Propose better instructions."*
5. Score the new ones. Keep the best. Repeat.

---

**[RUN]** = just execute. **[TODO]** = complete before running.

## 1) Setup

**[RUN]**

In [ ]:
!pip install -q transformers accelerate datasets
import torch
print(f"\u2705 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(10)+chr(10)+'  WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU.'}")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # half-precision: 2x faster, half the VRAM
    device_map="auto",           # auto-selects GPU
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
import re
import uuid, time, html as html_lib
from IPython.display import display, HTML


def clean_model_text(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()

    # Unwrap common LaTeX answer wrappers.
    # Examples: "\\boxed{42}", "$\\boxed{42}$", "boxed{42}", "$42$", "\\(42\\)", "\\[42\\]".
    t = re.sub(r"\$?\\\\boxed\{([^}]*)\}\$?", r"\1", t)
    t = re.sub(r"\$?boxed\{([^}]*)\}\$?", r"\1", t)

    m = re.fullmatch(r"\$([^$]+)\$", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\((.*)\\\)", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\[(.*)\\\]", t)
    if m:
        t = m.group(1).strip()

    return t


def generate_response(messages, max_new_tokens=None, creative=False):
    """Send chat-formatted messages to the model and return the response."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = dict(
        do_sample=creative,
        temperature=0.7 if creative else 1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    if max_new_tokens is not None:
        gen_kwargs["max_new_tokens"] = max_new_tokens

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return clean_model_text(response)


def display_sample(question, answer):
    display(HTML(
        f"<p><strong>Question:</strong> {question}</p>"
        f"<p><strong>Answer:</strong> {answer}</p>"
    ))


def display_response(prompt_text, response_text, elapsed=None):
    uid = str(uuid.uuid4()).replace("-", "")
    time_tag = f"<div style='color:#666;font-size:90%;margin-top:4px'>&#x23F1; {elapsed:.1f}s</div>" if elapsed else ""
    safe_p = html_lib.escape(str(prompt_text).strip())
    safe_r = html_lib.escape(clean_model_text(str(response_text))).strip()
    display(HTML(
        "<style>.rt{border-collapse:collapse;width:100%;margin:10px 0;font-family:'Segoe UI',sans-serif;font-size:14px}"
        ".rt th,.rt td{border:1px solid #ddd;padding:9px 12px;vertical-align:top}"
        ".rt th{background:#f5f5f5;width:120px;font-weight:600}"
        "pre.rm{white-space:pre-wrap;margin:0}</style>"
        f"<table class='rt'>"
        f"<tr><th>Prompt</th><td><pre class='rm'>{safe_p}</pre></td></tr>"
        f"<tr><th>Response</th><td><pre class='rm'>{safe_r}</pre>{time_tag}</td></tr>"
        "</table>"
    ))


def run_and_display(messages, max_new_tokens=None, creative=False):
    """Generate and display in a table. Returns the response string."""
    prompt_text = messages[-1]["content"]
    t0 = time.time()
    response = generate_response(messages, max_new_tokens=max_new_tokens, creative=creative)
    elapsed = time.time() - t0
    display_response(prompt_text, response, elapsed)
    return response

## 2) Load GSM8K

**[RUN]** Tiny subsets keep each scoring round fast.

In [ ]:
import re
from tqdm import tqdm
from typing import List, Dict, Tuple

N_TRAIN = 3   # questions scored per instruction during the OPRO loop
N_EVAL  = 3   # held-out questions for final validation

ds_main = load_dataset("openai/gsm8k", "main")
train_subset = ds_main["train"].shuffle(seed=123).select(range(N_TRAIN))
eval_subset  = ds_main["test"].shuffle(seed=321).select(range(N_EVAL))
print(f"Train subset: {N_TRAIN} | Eval subset: {N_EVAL}")

## 3) Utility Functions

**[RUN]** Quick note on the generation function used for scoring:

In [ ]:
# generate_response is defined above in the shared utility cell.
# Quick reminder of the two modes we use in this notebook:
#
#   generate_response(messages, max_new_tokens=50,  creative=False)  <- scoring (math answers are short)
#   generate_response(messages, max_new_tokens=200, creative=False)  <- meta-prompt (stable proposals)
print("generate_response ready.")

**[RUN + TODO]** Three helper functions:

1. **`extract_final_number`** — GSM8K stores gold answers after `####`. Already done.
2. **[TODO] `gsm8k_gold_answer`** — uses (1) to get the correct answer from a dataset row.
3. **[TODO] `accuracy`** — fraction of predictions that exactly match gold answers.

In [ ]:
def extract_final_number(text: str) -> str:
    """Extract the final answer number. GSM8K gold answers follow '#### <number>'."""
    if not text:
        return ""
    m = re.search(r"####\s*([-+]?\d+(?:\.\d+)?)", text)
    if m:
        return m.group(1)
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else ""

def gsm8k_gold_answer(example: Dict) -> str:
    # TODO: use extract_final_number to pull the answer from example["answer"]
    pass

def accuracy(preds: List[str], golds: List[str]) -> float:
    # TODO: return fraction of preds that match golds (avoid division by zero)
    pass

## 4) Seed Instructions

**[RUN]** Two handcrafted instructions as the starting point. OPRO will try to beat them.

> 🎛️ Feel free to change these. Keep the tuple format `(text, label)`.

In [ ]:
SEED_INSTRUCTIONS = [
    ("Solve the math word problem. Show your reasoning and end with '#### <number>'.", "Seed-1"),
    ("Think step by step. End your answer with '#### <number>'.", "Seed-2"),
]
print("Seed instructions ready.")

## 5) Prompting Templates

**[TODO]** Two templates:

1. **Task prompt** — wraps an instruction + question for the math solver.
2. **Meta-prompt** — shows the leaderboard, asks for K better instructions.

`build_task_messages` is done. Complete `build_meta_prompt` by returning the message list.

> 💡 Look at how `build_task_messages` builds its return value — same structure.

In [ ]:
TASK_TMPL = "Instruction: {instruction}\n\nQuestion: {question}\n\nAnswer (end with #### <number>):"

def build_task_messages(instruction: str, question: str) -> List[Dict]:
    return [
        {"role": "system", "content": "You are a helpful math solver."},
        {"role": "user",   "content": TASK_TMPL.format(instruction=instruction, question=question)},
    ]

def build_meta_prompt(scored_instructions: List[Tuple[str, float, str]], K: int = 3) -> List[Dict]:
    """OPRO meta-prompt: show the leaderboard, ask for K better instructions."""
    ranked = sorted(scored_instructions, key=lambda x: x[1], reverse=True)
    table  = "\n".join(
        f"  [{i+1}] score={s:.2f} | {id_}\n       {instr}"
        for i, (instr, s, id_) in enumerate(ranked)
    )
    meta_user = (
        f"You are optimizing instructions for solving GSM8K math problems.\n"
        f"Higher accuracy score = better instruction.\n\n"
        f"Current leaderboard:\n{table}\n\n"
        f"Propose {K} NEW instructions that could score higher.\n"
        f"Rules:\n"
        f"- End each with a reminder to output '#### <number>'.\n"
        f"- Be concise (1-2 sentences each).\n"
        f"- Do not reword existing instructions.\n"
        f"- Return a numbered list 1..{K}, one per line."
    )
    # TODO: return a message list:
    #   [{"role": "system", "content": "You are an instruction optimizer."},
    #    {"role": "user",   "content": meta_user}]
    return []

def parse_candidates(text: str, K: int) -> List[str]:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    cands = []
    for ln in lines:
        m = re.match(r"^\d+[.\)\s]\s*(.*)", ln)
        if m:
            cands.append(m.group(1).strip())
    return (cands or [text.strip()])[:K]

## 6) Scoring Function

**[TODO]** `score_instruction` turns any instruction into an accuracy number.

Four steps:
1. Build task messages per question.
2. Call `generate_response` with `max_new_tokens=50, creative=False` (math answers are short).
3. Extract predicted and gold numbers.
4. Return `accuracy(preds, golds)`.

In [ ]:
MAX_Q = N_TRAIN

def score_instruction(instruction: str, dataset, max_q: int = MAX_Q) -> float:
    """Score an instruction on max_q questions and return accuracy."""
    preds, golds = [], []
    for ex in tqdm(dataset.select(range(min(len(dataset), max_q))), leave=False):
        # TODO 1: messages = build_task_messages(instruction, ex["question"])
        messages = []

        # TODO 2: out = generate_response(messages, max_new_tokens=50, creative=False)
        #         (50 tokens is enough for a math answer)
        out = ""

        # TODO 3: pred = extract_final_number(out)
        #         gold = gsm8k_gold_answer(ex)
        pred, gold = "", ""

        preds.append(pred)
        golds.append(gold)

    # TODO 4: return accuracy(preds, golds)
    pass

## Tie-Breaking

**[RUN]** When two instructions tie on accuracy, prefer non-seeds (novelty) and shorter text (concision).

In [ ]:
def is_seed(id_): return id_.startswith("Seed")
def tie_key(item):
    instr, s, id_ = item
    return (s, 1 if not is_seed(id_) else 0, -len(instr))

## 7) The OPRO Loop

**[TODO]** The bookkeeping is done. Two tasks:
1. `meta_msgs = build_meta_prompt(scored, K=K_NEW)`
2. `meta_out = generate_response(meta_msgs, max_new_tokens=200, creative=False)`

In [ ]:
ROUNDS   = 1
K_NEW    = 3
TOP_KEEP = 3

scored: List[Tuple[str, float, str]] = []
for instr, id_ in SEED_INSTRUCTIONS:
    s = score_instruction(instr, train_subset)
    print(f"  {id_}: accuracy = {s:.2f}")
    scored.append((instr, s, id_))

for r in range(1, ROUNDS + 1):
    print(f"\n=== OPRO Round {r}/{ROUNDS} ===")

    # TODO 1: meta_msgs = build_meta_prompt(scored, K=K_NEW)
    meta_msgs = []

    # TODO 2: meta_out = generate_response(meta_msgs, max_new_tokens=200, creative=False)
    meta_out = ""

    candidates = parse_candidates(meta_out, K=K_NEW)
    existing   = {x[0] for x in scored}
    fresh      = [c for c in candidates if c not in existing]

    print(f"Proposed {len(fresh)} new instructions:")
    for c in fresh:
        print(f"  -> {c[:80]}")

    for i, cand in enumerate(fresh):
        s   = score_instruction(cand, train_subset)
        cid = f"r{r}-{i+1}"
        print(f"  {cid}: accuracy = {s:.2f}")
        scored.append((cand, s, cid))

    scored = sorted(scored, key=tie_key, reverse=True)[:TOP_KEEP]
    print("Top so far:")
    for instr, s, id_ in scored:
        print(f"  {id_} ({s:.2f}): {instr[:70]}")

## 8) Final Evaluation

**[RUN]** Test the best instruction on held-out data.

In [ ]:
best_instr, best_score, best_id = max(scored, key=tie_key)
print(f"Best instruction ({best_id}, train accuracy {best_score:.2f}):")
print(f"  {best_instr}\n")
final_acc = score_instruction(best_instr, eval_subset, max_q=N_EVAL)
print(f"Held-out accuracy: {final_acc:.2f}")

# 🔄 How the Pieces Fit Together

You just ran automated prompt engineering:

1. **Seeded** with two handcrafted instructions.
2. **Scored** each on a tiny training slice.
3. **Built a meta-prompt** showing the leaderboard and asking for improvements.
4. **Scored** the model's proposals the same way.
5. **Kept the best** with tie-breaking that prefers novelty and concision.
6. **Validated** on held-out data.

This is **OPRO** in under 100 lines. Scale up `N_TRAIN`, `ROUNDS`, and `K_NEW` for more robust optimisation. 🎉